In [51]:
import optuna
import numpy as np
import pandas as pd
import wandb
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier

import joblib
import os

wandb.login(
    key="6b79e7cde2ff5d27341e1f12820cf0725b4d1f99",
    relogin=True
)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\kysal\_netrc


True

In [52]:
iris = load_iris()
X = iris.data
y = iris.target


In [53]:
wandb.init(
    project="iris-classification-mlops",
    name="optuna-hyperparameter-search",
    config={
        "model": "RandomForestClassifier",
        "dataset": "Iris"
    }
)


In [54]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "random_state": 42
    }

    model = RandomForestClassifier(**params)

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    ).mean()

    # W&B log
    wandb.log({
        "accuracy": score,
        **params
    })

    return score

In [55]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)


[I 2026-01-05 14:30:21,154] A new study created in memory with name: no-name-fdb0c9dc-21c6-45b3-aaf4-aad54e717101
[I 2026-01-05 14:30:22,338] Trial 0 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 258, 'max_depth': 20, 'min_samples_split': 2}. Best is trial 0 with value: 0.9666666666666668.
[I 2026-01-05 14:30:22,746] Trial 1 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 98, 'max_depth': 6, 'min_samples_split': 8}. Best is trial 0 with value: 0.9666666666666668.
[I 2026-01-05 14:30:23,095] Trial 2 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 84, 'max_depth': 7, 'min_samples_split': 6}. Best is trial 0 with value: 0.9666666666666668.
[I 2026-01-05 14:30:23,477] Trial 3 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 89, 'max_depth': 12, 'min_samples_split': 8}. Best is trial 0 with value: 0.9666666666666668.
[I 2026-01-05 14:30:24,040] Trial 4 finished with value: 0.9666666666666668

In [56]:
print("Best score:", study.best_value)
print("Best params:", study.best_params)
wandb.log({
    "best_accuracy": study.best_value,
    **study.best_params
})


Best score: 0.9666666666666668
Best params: {'n_estimators': 258, 'max_depth': 20, 'min_samples_split': 2}


In [57]:
best_model = RandomForestClassifier(
    **study.best_params,
    random_state=42
)

best_model.fit(X, y)


RandomForestClassifier(max_depth=20, n_estimators=258, random_state=42)

In [58]:
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "optuna_best_model.pkl")

joblib.dump(best_model, MODEL_PATH)

MODEL_PATH


'../models\\optuna_best_model.pkl'

In [59]:
os.path.exists(MODEL_PATH)

True

In [60]:
wandb.finish()

accuracy,███████████▁█▆██▆██████████▆▁█
best_accuracy,▁
max_depth,█▃▃▅▂▄▇▅▅▄█▁▆▃█▆▃▆▂▃▁▃▇▃▂▄▅▃▁▂█
min_samples_split,▁▆▅▆▅▇▄▃▃█▁▁▆▆▅▂█▅▄▂▇▅▅▄▇▂▃▅▅▆▁
n_estimators,▇▂▂▂▃▇▅▃▄▇█▅▁▆▄▆█▂▅▃▁▂▂▃▄▂▃▇▆▁▇
random_state,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.96667
best_accuracy,0.96667
max_depth,20
min_samples_split,2
n_estimators,258
